## Step 1: Environment Setup & GPU Configuration

In [1]:
# Install required packages
print("📦 Installing required packages...")
!pip install -q wandb pytorch-msssim lpips rasterio scikit-learn tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import vgg19
import os
import pandas as pd
import json
import glob
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# Check GPU availability
print(f"\n{'='*70}")
print("GPU CONFIGURATION")
print(f"{'='*70}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    device = torch.device('cuda:0')
else:
    print("⚠️  WARNING: No GPU detected! Training will be very slow.")
    device = torch.device('cpu')
print(f"Using device: {device}")
print(f"{'='*70}\n")

# Set up output directory
OUTPUT_DIR = '/kaggle/working/RFB-ESRGAN-Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory: {OUTPUT_DIR}")

📦 Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 60.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━

## Step 2: Initialize WandB

In [2]:
import wandb

# Login with WandB API key
wandb.login(key="5424a3d65aac1662f5be82d4439aaac35046689e")
print("✓ WandB authentication successful!")

# Initialize WandB run
wandb.init(
    project="agricultural-sr",
    name="resume-160k-kaggle-dual-gpu",
    config={
        "platform": "kaggle",
        "gpus": torch.cuda.device_count(),
        "architecture": "RFB-ESRGAN",
        "start_iteration": 160000,
        "target_iteration": 200000,
        "num_rrdb": 12,
        "num_rrfdb": 6,
        "num_feat": 64,
        "batch_size": 8,  # Adjust based on GPU memory
        "lr": 1e-4,
        "lr_size": 32,
        "hr_size": 256
    },
    resume="allow"
)

print(f"\n✓ WandB run initialized: {wandb.run.name}")
print(f"  Project: {wandb.run.project}")
print(f"  URL: {wandb.run.url}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hegdesudarshan (hegdesudarshan-hegde) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB authentication successful!



✓ WandB run initialized: resume-160k-kaggle-dual-gpu
  Project: agricultural-sr
  URL: https://wandb.ai/hegdesudarshan-hegde/agricultural-sr/runs/fluq9at6


## Step 3: Model Architecture Definitions

In [ ]:
# ========== RRDB (Residual in Residual Dense Block) ==========
class DenseBlock(nn.Module):
    def __init__(self, nf=64, gc=32):
        super(DenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, nf):
        super(RRDB, self).__init__()
        self.db1 = DenseBlock(nf)
        self.db2 = DenseBlock(nf)
        self.db3 = DenseBlock(nf)

    def forward(self, x):
        out = self.db1(x)
        out = self.db2(out)
        out = self.db3(out)
        return out * 0.2 + x

# ========== RFB (Receptive Field Block) - Match checkpoint structure ==========
class RFB(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(RFB, self).__init__()
        # Split channels properly: for 64 channels -> 21, 21, 22 to sum to 64
        branch_channels = out_channels // 3
        remaining = out_channels - (branch_channels * 2)  # Third branch gets remainder
        
        # Use nn.Sequential with explicit indexing to match checkpoint structure
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, 1),  # index 0
            nn.ReLU(inplace=True),  # index 1
            nn.Conv2d(branch_channels, branch_channels, 3, padding=1),  # index 2
            nn.ReLU(inplace=True)  # index 3
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, 1),  # index 0
            nn.ReLU(inplace=True),  # index 1
            nn.Conv2d(branch_channels, branch_channels, 3, padding=2, dilation=2),  # index 2
            nn.ReLU(inplace=True)  # index 3
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, remaining, 1),  # index 0 - uses remaining channels
            nn.ReLU(inplace=True),  # index 1
            nn.Conv2d(remaining, remaining, 3, padding=3, dilation=3),  # index 2
            nn.ReLU(inplace=True)  # index 3
        )
        # Match checkpoint: conv_concat.0.weight
        self.conv_concat = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1)
        )

    def forward(self, x):
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = torch.cat([x1, x2, x3], dim=1)
        return self.conv_concat(x_cat) + x

class RRFDB(nn.Module):
    def __init__(self, nf):
        super(RRFDB, self).__init__()
        # Checkpoint has 5 RFB blocks per RRFDB
        self.rfb1 = RFB(nf, nf)
        self.rfb2 = RFB(nf, nf)
        self.rfb3 = RFB(nf, nf)
        self.rfb4 = RFB(nf, nf)
        self.rfb5 = RFB(nf, nf)

    def forward(self, x):
        out = self.rfb1(x)
        out = self.rfb2(out)
        out = self.rfb3(out)
        out = self.rfb4(out)
        out = self.rfb5(out)
        return out * 0.2 + x

# ========== GENERATOR - Match checkpoint structure ==========
class Generator(nn.Module):
    def __init__(self, num_rrdb=12, num_rrfdb=6, nf=64, scale=8):
        super(Generator, self).__init__()
        self.conv_first = nn.Conv2d(3, nf, 3, 1, 1)
        
        # RRDB blocks - named trunk_a in checkpoint
        self.trunk_a = nn.Sequential(*[RRDB(nf) for _ in range(num_rrdb)])
        
        # RRFDB blocks - named trunk_rfb in checkpoint
        self.trunk_rfb = nn.Sequential(*[RRFDB(nf) for _ in range(num_rrfdb)])
        
        # RFB upsampling - named rfb_up in checkpoint
        self.rfb_up = RFB(nf, nf)
        
        # Upsampling (32x32 -> 256x256 = 8x) - match checkpoint structure
        self.upsample = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),  # index 0
            nn.LeakyReLU(0.2, inplace=True),  # index 1
            nn.PixelShuffle(2),  # index 2
            nn.Conv2d(nf, nf * 4, 3, 1, 1),  # index 3
            nn.LeakyReLU(0.2, inplace=True),  # index 4
            nn.PixelShuffle(2),  # index 5
            nn.Conv2d(nf, nf * 4, 3, 1, 1),  # index 6
            nn.LeakyReLU(0.2, inplace=True),  # index 7
            nn.PixelShuffle(2)  # index 8
        )
        
        # Final convolutions - named conv_final in checkpoint
        self.conv_final = nn.Sequential(
            nn.Conv2d(nf, nf, 3, 1, 1),  # index 0
            nn.LeakyReLU(0.2, inplace=True),  # index 1
            nn.Conv2d(nf, 3, 3, 1, 1)  # index 2
        )

    def forward(self, x):
        fea = self.conv_first(x)
        trunk_a_out = self.trunk_a(fea)
        trunk_rfb_out = self.trunk_rfb(trunk_a_out)
        rfb_up_out = self.rfb_up(trunk_rfb_out)
        fea = fea + rfb_up_out
        
        fea = self.upsample(fea)
        out = self.conv_final(fea)
        return out

# ========== DISCRIMINATOR ==========
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.Conv2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(512, 1024, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(1024, 1, 1)
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)

print("✓ Model architectures defined")

✓ Model architectures defined


## Step 4: Dataset Setup

In [14]:
# Define dataset paths
DATASET_ROOT = '/kaggle/input/bigearthnetv2-s2-4/BigEarthNet-S2'
TRAIN_CSV = '/kaggle/input/label-indices/train.csv'
VAL_CSV = '/kaggle/input/label-indices/val.csv'

print(f"{'='*70}")
print("DATASET CONFIGURATION")
print(f"{'='*70}")
print(f"BigEarthNet Root: {DATASET_ROOT}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Val CSV: {VAL_CSV}")

# Verify paths exist
assert os.path.exists(DATASET_ROOT), f"Dataset not found: {DATASET_ROOT}"
print(f"✓ All paths verified")

# Build patch index
print(f"\n🗂️  Building patch index...")
root_contents = os.listdir(DATASET_ROOT)
all_dirs = [d for d in root_contents if os.path.isdir(os.path.join(DATASET_ROOT, d))]

patch_to_path = {}
for tile_dir in tqdm(all_dirs, desc="Indexing tiles", ncols=80):
    tile_path = os.path.join(DATASET_ROOT, tile_dir)
    try:
        for patch_name in os.listdir(tile_path):
            patch_path = os.path.join(tile_path, patch_name)
            if os.path.isdir(patch_path):
                patch_to_path[patch_name] = patch_path
    except:
        continue

print(f"   ✓ Indexed {len(patch_to_path):,} patches")

# Create train/val split from available patches
all_patch_names = list(patch_to_path.keys())
train_patches, val_patches = train_test_split(
    all_patch_names, test_size=0.2, random_state=42
)

train_df = pd.DataFrame({'patch_name': train_patches})
val_df = pd.DataFrame({'patch_name': val_patches})

print(f"\n✓ Dataset split:")
print(f"  Training: {len(train_df):,} patches")
print(f"  Validation: {len(val_df):,} patches")
print(f"{'='*70}\n")

DATASET CONFIGURATION
BigEarthNet Root: /kaggle/input/bigearthnetv2-s2-4/BigEarthNet-S2
Train CSV: /kaggle/input/label-indices/train.csv
Val CSV: /kaggle/input/label-indices/val.csv
✓ All paths verified

🗂️  Building patch index...


Indexing tiles:   0%|                                    | 0/10 [00:00<?, ?it/s]

   ✓ Indexed 28,937 patches

✓ Dataset split:
  Training: 23,149 patches
  Validation: 5,788 patches



In [15]:
# TIF loading function
def load_rgb_from_tif(patch_path, target_size=256):
    """Load RGB bands from BigEarthNet TIF files."""
    import rasterio
    
    band_mapping = {'R': 'B04', 'G': 'B03', 'B': 'B02'}
    rgb_arrays = []
    
    for color, band_name in band_mapping.items():
        band_files = glob.glob(os.path.join(patch_path, f'*_{band_name}.tif'))
        if not band_files:
            raise FileNotFoundError(f"Band {band_name} not found in {patch_path}")
        
        with rasterio.open(band_files[0]) as src:
            band_data = src.read(1)
            band_data = np.clip(band_data / 10000.0 * 255, 0, 255).astype(np.uint8)
            rgb_arrays.append(band_data)
    
    rgb_image = np.stack(rgb_arrays, axis=-1)
    pil_image = Image.fromarray(rgb_image, mode='RGB')
    
    if pil_image.size != (target_size, target_size):
        pil_image = pil_image.resize((target_size, target_size), Image.BICUBIC)
    
    return pil_image

# Dataset class
class BigEarthNetDataset(Dataset):
    def __init__(self, dataframe, patch_index, lr_size=32, hr_size=256, transform=None):
        self.df = dataframe
        self.patch_index = patch_index
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patch_name = str(self.df.iloc[idx]['patch_name']).strip()
        patch_path = self.patch_index[patch_name]
        
        hr_img = load_rgb_from_tif(patch_path, target_size=self.hr_size)
        lr_img = hr_img.resize((self.lr_size, self.lr_size), Image.BICUBIC)
        
        if self.transform:
            lr_img = self.transform(lr_img)
            hr_img = self.transform(hr_img)
        
        return lr_img, hr_img

# Create datasets
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = BigEarthNetDataset(train_df, patch_to_path, transform=transform)
val_dataset = BigEarthNetDataset(val_df, patch_to_path, transform=transform)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=wandb.config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=wandb.config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n✓ Datasets created!")
print(f"  Training batches: {len(train_loader):,}")
print(f"  Validation batches: {len(val_loader):,}")

# Test loading
test_lr, test_hr = next(iter(train_loader))
print(f"\n🧪 Data loading test:")
print(f"  LR shape: {test_lr.shape}")
print(f"  HR shape: {test_hr.shape}")
print(f"  LR range: [{test_lr.min():.3f}, {test_lr.max():.3f}]")
print(f"  HR range: [{test_hr.min():.3f}, {test_hr.max():.3f}]")
print(f"\n✓ Data loading verified!")


✓ Datasets created!
  Training batches: 2,894
  Validation batches: 724

🧪 Data loading test:
  LR shape: torch.Size([8, 3, 32, 32])
  HR shape: torch.Size([8, 3, 256, 256])
  LR range: [-0.992, -0.631]
  HR range: [-1.000, -0.341]

✓ Data loading verified!


## Step 5: Initialize Models & Load Checkpoint

In [16]:
# Initialize models
print(f"\n{'='*70}")
print("INITIALIZING MODELS")
print(f"{'='*70}")

generator = Generator(
    num_rrdb=wandb.config.num_rrdb,
    num_rrfdb=wandb.config.num_rrfdb,
    nf=wandb.config.num_feat
)

discriminator = Discriminator()

# Load checkpoint
checkpoint_path = '/kaggle/input/generator-iter-160000-pth/pytorch/default/1/generator_iter_160000.pth'
print(f"\n📥 Loading checkpoint from iteration 160,000...")
print(f"   Path: {checkpoint_path}")

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f"Checkpoint not found: {checkpoint_path}\n"
        "Please ensure 'generator-iter-160000-pth' dataset is added as input."
    )

checkpoint_state = torch.load(checkpoint_path, map_location='cpu')

# Debug: Print first 10 keys from each
print(f"\n🔍 Debug - Model keys (first 10):")
model_keys = list(generator.state_dict().keys())
for i, key in enumerate(model_keys[:10]):
    print(f"   {i+1}. {key}")

print(f"\n🔍 Debug - Checkpoint keys (first 10):")
checkpoint_keys = list(checkpoint_state.keys())
for i, key in enumerate(checkpoint_keys[:10]):
    print(f"   {i+1}. {key}")

# Try to load with strict=False to see what happens
missing_keys, unexpected_keys = generator.load_state_dict(checkpoint_state, strict=False)
print(f"\n⚠️  Missing keys: {len(missing_keys)}")
if len(missing_keys) > 0:
    print(f"   First missing: {missing_keys[0]}")
print(f"⚠️  Unexpected keys: {len(unexpected_keys)}")
if len(unexpected_keys) > 0:
    print(f"   First unexpected: {unexpected_keys[0]}")

if len(missing_keys) == 0 and len(unexpected_keys) == 0:
    print(f"✓ Checkpoint loaded successfully!")

# Setup for dual GPU
if torch.cuda.device_count() > 1:
    print(f"\n🚀 Using {torch.cuda.device_count()} GPUs with DataParallel")
    generator = nn.DataParallel(generator)
    discriminator = nn.DataParallel(discriminator)
    print(f"   Generator replicated across GPUs")
    print(f"   Discriminator replicated across GPUs")

generator = generator.to(device)
discriminator = discriminator.to(device)

# Print model info
gen_params = sum(p.numel() for p in generator.parameters()) / 1e6
disc_params = sum(p.numel() for p in discriminator.parameters()) / 1e6
print(f"\n📊 Model Statistics:")
print(f"   Generator parameters: {gen_params:.2f}M")
print(f"   Discriminator parameters: {disc_params:.2f}M")
print(f"   Total parameters: {gen_params + disc_params:.2f}M")
print(f"{'='*70}\n")


INITIALIZING MODELS

📥 Loading checkpoint from iteration 160,000...
   Path: /kaggle/input/generator-iter-160000-pth/pytorch/default/1/generator_iter_160000.pth

🔍 Debug - Model keys (first 10):
   1. conv_first.weight
   2. conv_first.bias
   3. trunk_a.0.db1.conv1.weight
   4. trunk_a.0.db1.conv1.bias
   5. trunk_a.0.db1.conv2.weight
   6. trunk_a.0.db1.conv2.bias
   7. trunk_a.0.db1.conv3.weight
   8. trunk_a.0.db1.conv3.bias
   9. trunk_a.0.db1.conv4.weight
   10. trunk_a.0.db1.conv4.bias

🔍 Debug - Checkpoint keys (first 10):
   1. conv_first.weight
   2. conv_first.bias
   3. trunk_a.0.db1.conv1.weight
   4. trunk_a.0.db1.conv1.bias
   5. trunk_a.0.db1.conv2.weight
   6. trunk_a.0.db1.conv2.bias
   7. trunk_a.0.db1.conv3.weight
   8. trunk_a.0.db1.conv3.bias
   9. trunk_a.0.db1.conv4.weight
   10. trunk_a.0.db1.conv4.bias

⚠️  Missing keys: 372
   First missing: trunk_rfb.0.rfb1.branch1.0.weight
⚠️  Unexpected keys: 372
   First unexpected: trunk_rfb.0.rfb1.branch1.1.weight

🚀 U

## Step 6: Loss Functions

In [17]:
import lpips
from pytorch_msssim import ms_ssim

# Perceptual loss (VGG)
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg = vgg19(pretrained=True).features
        self.layers = nn.Sequential(*list(vgg)[:35]).eval()
        for param in self.layers.parameters():
            param.requires_grad = False
        
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
    
    def forward(self, x, y):
        x = (x * 0.5 + 0.5 - self.mean) / self.std
        y = (y * 0.5 + 0.5 - self.mean) / self.std
        x_feat = self.layers(x)
        y_feat = self.layers(y)
        return F.mse_loss(x_feat, y_feat)

# Initialize losses
criterion_pixel = nn.L1Loss().to(device)
criterion_perceptual = VGGPerceptualLoss().to(device)
criterion_gan = nn.BCEWithLogitsLoss().to(device)
lpips_loss_fn = lpips.LPIPS(net='alex').to(device)

print("✓ Loss functions initialized")

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 211MB/s] 


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100%|██████████| 233M/233M [00:01<00:00, 229MB/s] 


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
✓ Loss functions initialized


## Step 7: Training Function

In [18]:
def train_stage2_resume(
    generator,
    discriminator,
    train_loader,
    val_loader,
    total_iterations=200000,
    start_iter=160000,
    lr=1e-4
):
    """
    Stage 2 GAN training with dual GPU support.
    """
    print(f"\n{'='*70}")
    print("STAGE 2: RESUMING GAN TRAINING")
    print(f"{'='*70}")
    print(f"Starting from: {start_iter:,}")
    print(f"Target: {total_iterations:,}")
    print(f"Remaining: {total_iterations - start_iter:,} iterations")
    print(f"{'='*70}\n")
    
    # Optimizers
    optimizer_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.9, 0.99))
    optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.9, 0.99))
    
    for param_group in optimizer_g.param_groups:
        param_group['initial_lr'] = lr
    for param_group in optimizer_d.param_groups:
        param_group['initial_lr'] = lr
    
    # LR schedulers
    milestones = [50000, 100000, 150000, 180000]
    scheduler_g = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_g, milestones=milestones, gamma=0.5, last_epoch=start_iter-1
    )
    scheduler_d = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_d, milestones=milestones, gamma=0.5, last_epoch=start_iter-1
    )
    
    # Training loop
    iteration = start_iter
    epoch = 0
    
    print(f"Starting training loop...\n")
    
    while iteration < total_iterations:
        epoch += 1
        generator.train()
        discriminator.train()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}", ncols=100)
        
        for lr_imgs, hr_imgs in pbar:
            if iteration >= total_iterations:
                break
            
            lr_imgs = lr_imgs.to(device)
            hr_imgs = hr_imgs.to(device)
            batch_size = lr_imgs.size(0)
            
            # Train Discriminator
            optimizer_d.zero_grad()
            
            sr_imgs = generator(lr_imgs)
            real_validity = discriminator(hr_imgs)
            fake_validity = discriminator(sr_imgs.detach())
            
            real_labels = torch.ones_like(real_validity)
            fake_labels = torch.zeros_like(fake_validity)
            
            d_loss_real = criterion_gan(real_validity, real_labels)
            d_loss_fake = criterion_gan(fake_validity, fake_labels)
            d_loss = (d_loss_real + d_loss_fake) / 2
            
            d_loss.backward()
            optimizer_d.step()
            
            # Train Generator
            optimizer_g.zero_grad()
            
            sr_imgs = generator(lr_imgs)
            fake_validity = discriminator(sr_imgs)
            
            # Generator losses
            g_loss_pixel = criterion_pixel(sr_imgs, hr_imgs)
            g_loss_perceptual = criterion_perceptual(sr_imgs, hr_imgs)
            g_loss_gan = criterion_gan(fake_validity, torch.ones_like(fake_validity))
            
            lpips_val = lpips_loss_fn(sr_imgs, hr_imgs).mean()
            ms_ssim_val = 1 - ms_ssim(sr_imgs, hr_imgs, data_range=2.0, size_average=True)
            
            g_loss = (
                g_loss_pixel * 1.0 +
                g_loss_perceptual * 0.1 +
                g_loss_gan * 0.005 +
                lpips_val * 0.1 +
                ms_ssim_val * 0.1
            )
            
            g_loss.backward()
            optimizer_g.step()
            
            iteration += 1
            
            # Update progress bar
            pbar.set_postfix({
                'iter': f'{iteration}',
                'g_loss': f'{g_loss.item():.4f}',
                'd_loss': f'{d_loss.item():.4f}'
            })
            
            # Log to WandB
            if iteration % 100 == 0:
                wandb.log({
                    'iteration': iteration,
                    'g_loss': g_loss.item(),
                    'd_loss': d_loss.item(),
                    'g_loss_pixel': g_loss_pixel.item(),
                    'g_loss_perceptual': g_loss_perceptual.item(),
                    'g_loss_gan': g_loss_gan.item(),
                    'lpips': lpips_val.item(),
                    'ms_ssim': ms_ssim_val.item(),
                    'lr_g': optimizer_g.param_groups[0]['lr'],
                    'lr_d': optimizer_d.param_groups[0]['lr']
                })
            
            # Save checkpoint
            if iteration % 5000 == 0:
                save_path = os.path.join(OUTPUT_DIR, f'generator_iter_{iteration}.pth')
                if isinstance(generator, nn.DataParallel):
                    torch.save(generator.module.state_dict(), save_path)
                else:
                    torch.save(generator.state_dict(), save_path)
                print(f"\n💾 Checkpoint saved: {save_path}")
            
            # Validation
            if iteration % 1000 == 0:
                generator.eval()
                val_losses = []
                
                with torch.no_grad():
                    for val_lr, val_hr in val_loader:
                        val_lr = val_lr.to(device)
                        val_hr = val_hr.to(device)
                        val_sr = generator(val_lr)
                        val_loss = criterion_pixel(val_sr, val_hr)
                        val_losses.append(val_loss.item())
                
                avg_val_loss = np.mean(val_losses)
                wandb.log({'val_loss': avg_val_loss, 'iteration': iteration})
                print(f"\n📊 Validation loss: {avg_val_loss:.4f}")
                
                generator.train()
        
        scheduler_g.step()
        scheduler_d.step()
    
    print(f"\n{'='*70}")
    print("✅ TRAINING COMPLETED!")
    print(f"{'='*70}")
    print(f"Final iteration: {iteration:,}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"{'='*70}\n")

print("✓ Training function defined")

✓ Training function defined


## Step 8: Start Training

In [19]:
# Start training
train_stage2_resume(
    generator=generator,
    discriminator=discriminator,
    train_loader=train_loader,
    val_loader=val_loader,
    total_iterations=200000,
    start_iter=160000,
    lr=1e-4
)


STAGE 2: RESUMING GAN TRAINING
Starting from: 160,000
Target: 200,000
Remaining: 40,000 iterations

Starting training loop...



Epoch 1:   0%|                                                             | 0/2894 [00:00<?, ?it/s]

RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/parallel_apply.py", line 96, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1750, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_38/3868908674.py", line 124, in forward
    trunk_rfb_out = self.trunk_rfb(trunk_a_out)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^

## Step 9: Save Final Model

In [ ]:
# Save final model
final_path = os.path.join(OUTPUT_DIR, 'generator_final_200000.pth')

if isinstance(generator, nn.DataParallel):
    torch.save(generator.module.state_dict(), final_path)
else:
    torch.save(generator.state_dict(), final_path)

print(f"\n✅ Final model saved: {final_path}")
print(f"\n📥 Download from: /kaggle/working/RFB-ESRGAN-Output/")

# Close WandB
wandb.finish()
print("\n✓ WandB run completed")

## Step 10: Comprehensive Model Evaluation

Now we'll perform extensive evaluation comparing our model against multiple baselines with 8 different metrics.

In [ ]:
# ========== COMPREHENSIVE EVALUATION METRICS ==========

# Install additional packages if needed
!pip install -q lpips pytorch-msssim scikit-learn seaborn scipy

import time
from collections import defaultdict
import seaborn as sns

print("✓ Evaluation packages installed!")

# ========== 1. SUPER-RESOLUTION METRICS ==========

class SuperResolutionMetrics:
    """Comprehensive SR evaluation metrics"""
    def __init__(self, device):
        self.device = device
        # LPIPS loss network (Alex)
        import lpips
        self.lpips_fn = lpips.LPIPS(net='alex').to(device)

    def calculate_psnr(self, sr, hr):
        """Peak Signal-to-Noise Ratio"""
        mse = F.mse_loss(sr, hr)
        psnr = 10 * torch.log10(4 / mse)  # Range [-1,1] → max=2, so 4
        return psnr.item()

    def calculate_ssim(self, sr, hr):
        """Structural Similarity Index"""
        from pytorch_msssim import ssim
        # Normalize from [-1,1] to [0,1]
        sr_norm = (sr + 1) / 2
        hr_norm = (hr + 1) / 2
        ssim_val = ssim(sr_norm, hr_norm, data_range=1.0, size_average=True)
        return ssim_val.item()

    def calculate_ms_ssim(self, sr, hr):
        """Multi-Scale Structural Similarity Index"""
        from pytorch_msssim import ms_ssim
        sr_norm = (sr + 1) / 2
        hr_norm = (hr + 1) / 2
        ms_ssim_val = ms_ssim(sr_norm, hr_norm, data_range=1.0, size_average=True)
        return ms_ssim_val.item()

    def calculate_lpips(self, sr, hr):
        """Learned Perceptual Image Patch Similarity"""
        lpips_val = self.lpips_fn(sr, hr)
        return lpips_val.mean().item()

    def calculate_mae(self, sr, hr):
        """Mean Absolute Error"""
        mae = F.l1_loss(sr, hr)
        return mae.item()

    def calculate_rmse(self, sr, hr):
        """Root Mean Square Error"""
        mse = F.mse_loss(sr, hr)
        rmse = torch.sqrt(mse)
        return rmse.item()

    def calculate_ndvi_error(self, sr, hr):
        """Spectral Consistency - NDVI Error for vegetation index accuracy"""
        # Extract red channel (assuming channel 0 is red after normalization)
        sr_red = sr[:, 0:1, :, :]  # Red channel
        hr_red = hr[:, 0:1, :, :]
        # Simplified NDVI approximation
        ndvi_error = F.l1_loss(sr_red, hr_red)
        return ndvi_error.item()

    def calculate_edge_preservation(self, sr, hr):
        """Edge Preservation using Sobel filters"""
        # Simple edge detection using convolution
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(self.device)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(self.device)

        # Average across channels
        sr_gray = sr.mean(dim=1, keepdim=True)
        hr_gray = hr.mean(dim=1, keepdim=True)

        # Apply Sobel filters
        sr_edge_x = F.conv2d(sr_gray, sobel_x, padding=1)
        sr_edge_y = F.conv2d(sr_gray, sobel_y, padding=1)
        hr_edge_x = F.conv2d(hr_gray, sobel_x, padding=1)
        hr_edge_y = F.conv2d(hr_gray, sobel_y, padding=1)

        sr_edge = torch.sqrt(sr_edge_x**2 + sr_edge_y**2)
        hr_edge = torch.sqrt(hr_edge_x**2 + hr_edge_y**2)

        edge_error = F.l1_loss(sr_edge, hr_edge)
        return edge_error.item()

    def evaluate_batch(self, sr, hr):
        """Evaluate all SR metrics on a batch"""
        metrics = {
            'psnr': self.calculate_psnr(sr, hr),
            'ssim': self.calculate_ssim(sr, hr),
            'ms_ssim': self.calculate_ms_ssim(sr, hr),
            'lpips': self.calculate_lpips(sr, hr),
            'mae': self.calculate_mae(sr, hr),
            'rmse': self.calculate_rmse(sr, hr),
            'ndvi_error': self.calculate_ndvi_error(sr, hr),
            'edge_preservation': self.calculate_edge_preservation(sr, hr)
        }
        return metrics


# ========== 2. BASELINE COMPARISON MODELS ==========

class BicubicUpsampler:
    """Baseline bicubic interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)


class BilinearUpsampler:
    """Baseline bilinear interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='bilinear', align_corners=False)


class NearestUpsampler:
    """Baseline nearest neighbor interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='nearest')


class SimpleSRCNN(nn.Module):
    """Lightweight SRCNN baseline for comparison"""
    def __init__(self, scale_factor=8):
        super(SimpleSRCNN, self).__init__()
        self.scale_factor = scale_factor
        # SRCNN: 3 conv layers
        self.conv1 = nn.Conv2d(3, 64, 9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, 1, padding=0)
        self.conv3 = nn.Conv2d(32, 3, 5, padding=2)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Bicubic upsampling first
        x = F.interpolate(x, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)
        return torch.tanh(x)


print("✓ Evaluation metrics and baseline models defined!")

In [ ]:
# ========== 3. COMPREHENSIVE COMPARATIVE EVALUATION ==========

def comparative_evaluation(generator, val_loader, device, num_samples=100):
    """Compare RFB-ESRGAN against multiple baselines with comprehensive metrics"""
    print("\n" + "="*70)
    print("COMPARATIVE EVALUATION: RFB-ESRGAN vs. Baselines")
    print("="*70)

    # Initialize models and metrics
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    bilinear = BilinearUpsampler(scale_factor=8)
    nearest = NearestUpsampler(scale_factor=8)
    srcnn = SimpleSRCNN(scale_factor=8).to(device)
    srcnn.eval()

    # Results storage
    model_names = ['nearest', 'bilinear', 'bicubic', 'srcnn', 'ours']
    results = {name: defaultdict(list) for name in model_names}
    inference_times = {name: [] for name in model_names}

    generator.eval()
    sample_count = 0

    print(f"\n📊 Evaluating on {num_samples} samples...")
    print(f"Models: Nearest, Bilinear, Bicubic, SRCNN, RFB-ESRGAN (Ours)")

    with torch.no_grad():
        for lr_img, hr_img in tqdm(val_loader, desc="Evaluating", ncols=80):
            if sample_count >= num_samples:
                break

            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            # ========== Nearest Neighbor ==========
            start_time = time.time()
            sr_nearest = nearest(lr_img)
            inference_times['nearest'].append(time.time() - start_time)
            metrics_nearest = sr_metrics.evaluate_batch(sr_nearest, hr_img)
            for k, v in metrics_nearest.items():
                results['nearest'][k].append(v)

            # ========== Bilinear ==========
            start_time = time.time()
            sr_bilinear = bilinear(lr_img)
            inference_times['bilinear'].append(time.time() - start_time)
            metrics_bilinear = sr_metrics.evaluate_batch(sr_bilinear, hr_img)
            for k, v in metrics_bilinear.items():
                results['bilinear'][k].append(v)

            # ========== Bicubic ==========
            start_time = time.time()
            sr_bicubic = bicubic(lr_img)
            inference_times['bicubic'].append(time.time() - start_time)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            for k, v in metrics_bicubic.items():
                results['bicubic'][k].append(v)

            # ========== SRCNN ==========
            start_time = time.time()
            sr_srcnn = srcnn(lr_img)
            inference_times['srcnn'].append(time.time() - start_time)
            metrics_srcnn = sr_metrics.evaluate_batch(sr_srcnn, hr_img)
            for k, v in metrics_srcnn.items():
                results['srcnn'][k].append(v)

            # ========== RFB-ESRGAN (Ours) ==========
            start_time = time.time()
            sr_ours = generator(lr_img)
            inference_times['ours'].append(time.time() - start_time)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            for k, v in metrics_ours.items():
                results['ours'][k].append(v)

            sample_count += lr_img.size(0)

    # ========== Calculate Average Metrics ==========
    print("\n" + "="*70)
    print("RESULTS SUMMARY")
    print("="*70)

    comparison_table = []

    for model_name in model_names:
        avg_metrics = {k: np.mean(v) for k, v in results[model_name].items()}
        avg_time = np.mean(inference_times[model_name]) * 1000  # ms

        print(f"\n{model_name.upper()}:")
        print(f"  PSNR: {avg_metrics['psnr']:.2f} dB")
        print(f"  SSIM: {avg_metrics['ssim']:.4f}")
        print(f"  MS-SSIM: {avg_metrics['ms_ssim']:.4f}")
        print(f"  LPIPS: {avg_metrics['lpips']:.4f} (lower is better)")
        print(f"  MAE: {avg_metrics['mae']:.4f}")
        print(f"  RMSE: {avg_metrics['rmse']:.4f}")
        print(f"  NDVI Error: {avg_metrics['ndvi_error']:.4f}")
        print(f"  Edge Preservation: {avg_metrics['edge_preservation']:.4f}")
        print(f"  Inference Time: {avg_time:.2f} ms/image")

        comparison_table.append({
            'model': model_name,
            **avg_metrics,
            'inference_time_ms': avg_time
        })

    # ========== Calculate Improvement Deltas ==========
    print("\n" + "="*70)
    print("IMPROVEMENT vs. BASELINES")
    print("="*70)

    ours_psnr = np.mean(results['ours']['psnr'])
    ours_ssim = np.mean(results['ours']['ssim'])
    bicubic_psnr = np.mean(results['bicubic']['psnr'])
    bicubic_ssim = np.mean(results['bicubic']['ssim'])
    srcnn_psnr = np.mean(results['srcnn']['psnr'])
    srcnn_ssim = np.mean(results['srcnn']['ssim'])

    delta_psnr_bicubic = ours_psnr - bicubic_psnr
    delta_ssim_bicubic = ours_ssim - bicubic_ssim
    delta_psnr_srcnn = ours_psnr - srcnn_psnr
    delta_ssim_srcnn = ours_ssim - srcnn_ssim

    print(f"\nΔPSNR vs. Bicubic: +{delta_psnr_bicubic:.2f} dB ({delta_psnr_bicubic/bicubic_psnr*100:.1f}% improvement)")
    print(f"ΔSSIM vs. Bicubic: +{delta_ssim_bicubic:.4f} ({delta_ssim_bicubic/bicubic_ssim*100:.1f}% improvement)")
    print(f"ΔPSNR vs. SRCNN: +{delta_psnr_srcnn:.2f} dB ({delta_psnr_srcnn/srcnn_psnr*100:.1f}% improvement)")
    print(f"ΔSSIM vs. SRCNN: +{delta_ssim_srcnn:.4f} ({delta_ssim_srcnn/srcnn_ssim*100:.1f}% improvement)")

    # ========== Model Efficiency ==========
    print("\n" + "="*70)
    print("MODEL EFFICIENCY METRICS")
    print("="*70)

    # Parameter count
    def count_parameters(model):
        if isinstance(model, nn.DataParallel):
            return sum(p.numel() for p in model.module.parameters())
        return sum(p.numel() for p in model.parameters())

    ours_params = count_parameters(generator)
    srcnn_params = count_parameters(srcnn)

    print(f"\nParameter Count:")
    print(f"  RFB-ESRGAN (Ours): {ours_params/1e6:.2f}M parameters")
    print(f"  SRCNN: {srcnn_params/1e6:.2f}M parameters")

    # Parameter efficiency
    psnr_per_param_ours = (ours_psnr - bicubic_psnr) / (ours_params / 1e6)
    psnr_per_param_srcnn = (srcnn_psnr - bicubic_psnr) / (srcnn_params / 1e6)

    print(f"\nParameter Efficiency (ΔPSNR per 1M params vs. Bicubic):")
    print(f"  RFB-ESRGAN: {psnr_per_param_ours:.3f} dB/M")
    print(f"  SRCNN: {psnr_per_param_srcnn:.3f} dB/M")

    # ========== System Performance Metrics ==========
    print("\n" + "="*70)
    print("SYSTEM PERFORMANCE METRICS")
    print("="*70)

    avg_time_ours = np.mean(inference_times['ours'])
    fps_ours = 1.0 / avg_time_ours if avg_time_ours > 0 else 0

    print(f"\nInference Performance (Ours):")
    print(f"  Latency: {avg_time_ours*1000:.2f} ms/image")
    print(f"  Throughput: {fps_ours:.2f} FPS")
    print(f"  Speed vs. Bicubic: {np.mean(inference_times['bicubic'])/avg_time_ours:.2f}x slower")

    # GPU Memory footprint
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = generator(lr_img)
        memory_allocated = torch.cuda.max_memory_allocated() / 1024**2  # MB
        print(f"  GPU Memory Footprint: {memory_allocated:.2f} MB")

    # ========== Statistical Significance ==========
    print("\n" + "="*70)
    print("STATISTICAL ANALYSIS")
    print("="*70)

    from scipy import stats

    # T-test comparing our model vs bicubic
    t_stat, p_value = stats.ttest_rel(results['ours']['psnr'], results['bicubic']['psnr'])
    print(f"\nPSNR T-test (Ours vs. Bicubic):")
    print(f"  t-statistic: {t_stat:.3f}")
    print(f"  p-value: {p_value:.6f}")
    print(f"  Statistically significant: {'Yes' if p_value < 0.05 else 'No'} (p < 0.05)")

    # Standard deviations
    print(f"\nStandard Deviations:")
    for model_name in ['bicubic', 'srcnn', 'ours']:
        std_psnr = np.std(results[model_name]['psnr'])
        std_ssim = np.std(results[model_name]['ssim'])
        print(f"  {model_name.upper()}: PSNR±{std_psnr:.2f}, SSIM±{std_ssim:.4f}")

    # ========== Log to WandB ==========
    wandb.log({
        'eval/psnr_ours': ours_psnr,
        'eval/ssim_ours': ours_ssim,
        'eval/ms_ssim_ours': np.mean(results['ours']['ms_ssim']),
        'eval/lpips_ours': np.mean(results['ours']['lpips']),
        'eval/delta_psnr_vs_bicubic': delta_psnr_bicubic,
        'eval/delta_ssim_vs_bicubic': delta_ssim_bicubic,
        'eval/inference_time_ms': avg_time_ours * 1000,
        'eval/throughput_fps': fps_ours,
        'eval/parameters_millions': ours_params / 1e6,
    })

    return comparison_table, results, inference_times


print("✓ Comparative evaluation function defined!")

In [ ]:
# ========== 4. VISUALIZATION FUNCTIONS ==========

def create_comparison_visualizations(results, inference_times, save_dir):
    """Create comprehensive comparison plots"""
    os.makedirs(save_dir, exist_ok=True)
    
    model_names = ['nearest', 'bilinear', 'bicubic', 'srcnn', 'ours']
    
    # Set style
    sns.set_style("whitegrid")
    colors = plt.cm.Set2(range(5))
    
    # ========== Figure 1: Main Metrics Bar Chart ==========
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # PSNR
    psnr_values = [np.mean(results[m]['psnr']) for m in model_names]
    axes[0, 0].bar(model_names, psnr_values, color=colors)
    axes[0, 0].set_ylabel('PSNR (dB)', fontsize=11)
    axes[0, 0].set_title('Peak Signal-to-Noise Ratio', fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # SSIM
    ssim_values = [np.mean(results[m]['ssim']) for m in model_names]
    axes[0, 1].bar(model_names, ssim_values, color=colors)
    axes[0, 1].set_ylabel('SSIM', fontsize=11)
    axes[0, 1].set_title('Structural Similarity Index', fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # MS-SSIM
    msssim_values = [np.mean(results[m]['ms_ssim']) for m in model_names]
    axes[1, 0].bar(model_names, msssim_values, color=colors)
    axes[1, 0].set_ylabel('MS-SSIM', fontsize=11)
    axes[1, 0].set_title('Multi-Scale SSIM', fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # LPIPS
    lpips_values = [np.mean(results[m]['lpips']) for m in model_names]
    axes[1, 1].bar(model_names, lpips_values, color=colors)
    axes[1, 1].set_ylabel('LPIPS (lower is better)', fontsize=11)
    axes[1, 1].set_title('Perceptual Distance (LPIPS)', fontweight='bold')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'metrics_comparison.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    # ========== Figure 2: Box Plots for Metric Distributions ==========
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # PSNR distribution
    psnr_data = [results[m]['psnr'] for m in model_names]
    bp1 = axes[0].boxplot(psnr_data, labels=model_names, patch_artist=True)
    for patch, color in zip(bp1['boxes'], colors):
        patch.set_facecolor(color)
    axes[0].set_ylabel('PSNR (dB)', fontsize=11)
    axes[0].set_title('PSNR Distribution', fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # SSIM distribution
    ssim_data = [results[m]['ssim'] for m in model_names]
    bp2 = axes[1].boxplot(ssim_data, labels=model_names, patch_artist=True)
    for patch, color in zip(bp2['boxes'], colors):
        patch.set_facecolor(color)
    axes[1].set_ylabel('SSIM', fontsize=11)
    axes[1].set_title('SSIM Distribution', fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    # LPIPS distribution
    lpips_data = [results[m]['lpips'] for m in model_names]
    bp3 = axes[2].boxplot(lpips_data, labels=model_names, patch_artist=True)
    for patch, color in zip(bp3['boxes'], colors):
        patch.set_facecolor(color)
    axes[2].set_ylabel('LPIPS', fontsize=11)
    axes[2].set_title('LPIPS Distribution (lower is better)', fontweight='bold')
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'metrics_distribution.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    # ========== Figure 3: Radar Chart ==========
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
    
    # Normalize metrics to 0-1 for radar chart
    categories = ['PSNR', 'SSIM', 'MS-SSIM', 'Edge\nPreserv.', 'Speed']
    
    for i, model_name in enumerate(['bicubic', 'srcnn', 'ours']):
        values = [
            np.mean(results[model_name]['psnr']) / 35.0,  # Normalize to ~35 dB max
            np.mean(results[model_name]['ssim']),
            np.mean(results[model_name]['ms_ssim']),
            np.mean(results[model_name]['edge_preservation']),
            1.0 / (np.mean(inference_times[model_name]) * 100)  # Inverse time
        ]
        
        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
        values += values[:1]
        angles += angles[:1]
        
        ax.plot(angles, values, 'o-', linewidth=2, label=model_name.upper(), color=colors[model_names.index(model_name)])
        ax.fill(angles, values, alpha=0.15, color=colors[model_names.index(model_name)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title('Multi-Metric Performance Comparison', fontsize=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.grid(True)
    
    fig_path = os.path.join(save_dir, 'radar_chart.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    # ========== Figure 4: Quality-Speed Tradeoff ==========
    plt.figure(figsize=(10, 6))
    
    for i, model_name in enumerate(model_names):
        avg_psnr = np.mean(results[model_name]['psnr'])
        avg_time = np.mean(inference_times[model_name]) * 1000  # ms
        
        plt.scatter(avg_time, avg_psnr, s=200, color=colors[i], label=model_name.upper(), alpha=0.7, edgecolors='black')
        plt.annotate(model_name.upper(), (avg_time, avg_psnr), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    plt.xlabel('Inference Time (ms/image)', fontsize=11)
    plt.ylabel('PSNR (dB)', fontsize=11)
    plt.title('Quality-Speed Tradeoff', fontsize=13, fontweight='bold')
    plt.legend(loc='best')
    plt.grid(alpha=0.3)
    
    fig_path = os.path.join(save_dir, 'quality_speed_tradeoff.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    plt.close()
    
    print(f"\n✓ All comparison visualizations saved to: {save_dir}")


print("✓ Visualization functions defined!")

In [ ]:
# ========== 5. VISUAL QUALITY COMPARISON ==========

def visualize_quality_comparison(generator, val_loader, device, save_dir, num_samples=5):
    """Generate side-by-side visual comparisons"""
    os.makedirs(save_dir, exist_ok=True)
    
    bicubic = BicubicUpsampler(scale_factor=8)
    sr_metrics = SuperResolutionMetrics(device)
    
    generator.eval()
    sample_idx = 0
    
    with torch.no_grad():
        for lr_img, hr_img in val_loader:
            if sample_idx >= num_samples:
                break
            
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate SR images
            sr_bicubic = bicubic(lr_img)
            sr_ours = generator(lr_img)
            
            # Calculate metrics
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            
            # Visualize first image in batch
            fig, axes = plt.subplots(1, 4, figsize=(18, 5))
            
            # LR (upsampled for visualization)
            lr_upsampled = F.interpolate(lr_img, scale_factor=8, mode='nearest')
            lr_np = lr_upsampled[0].cpu().permute(1, 2, 0).numpy()
            lr_np = np.clip(lr_np, 0, 1)
            axes[0].imshow(lr_np)
            axes[0].set_title(f'LR Input\n(×8 nearest)', fontsize=11, fontweight='bold')
            axes[0].axis('off')
            
            # HR (Ground Truth)
            hr_np = hr_img[0].cpu().permute(1, 2, 0).numpy()
            hr_np = np.clip(hr_np, 0, 1)
            axes[1].imshow(hr_np)
            axes[1].set_title('HR Ground Truth', fontsize=11, fontweight='bold')
            axes[1].axis('off')
            
            # Bicubic
            bicubic_np = sr_bicubic[0].cpu().permute(1, 2, 0).numpy()
            bicubic_np = np.clip(bicubic_np, 0, 1)
            axes[2].imshow(bicubic_np)
            axes[2].set_title(f'Bicubic\nPSNR: {metrics_bicubic["psnr"]:.2f} dB\nSSIM: {metrics_bicubic["ssim"]:.3f}', 
                            fontsize=10, fontweight='bold')
            axes[2].axis('off')
            
            # Ours
            ours_np = sr_ours[0].cpu().permute(1, 2, 0).numpy()
            ours_np = np.clip(ours_np, 0, 1)
            axes[3].imshow(ours_np)
            axes[3].set_title(f'RFB-ESRGAN (Ours)\nPSNR: {metrics_ours["psnr"]:.2f} dB\nSSIM: {metrics_ours["ssim"]:.3f}', 
                            fontsize=10, fontweight='bold', color='green')
            axes[3].axis('off')
            
            plt.tight_layout()
            fig_path = os.path.join(save_dir, f'comparison_sample_{sample_idx+1}.png')
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            print(f"✓ Saved: comparison_sample_{sample_idx+1}.png")
            plt.close()
            
            sample_idx += 1
    
    print(f"\n✓ Quality comparison visualizations saved to: {save_dir}")


print("✓ Visual quality comparison function defined!")

In [ ]:
# ========== 6. DIFFERENCE MAPS VISUALIZATION ==========

def visualize_difference_maps(generator, val_loader, device, save_dir, num_samples=5):
    """Visualize pixel-wise error maps"""
    os.makedirs(save_dir, exist_ok=True)
    
    bicubic = BicubicUpsampler(scale_factor=8)
    
    generator.eval()
    sample_idx = 0
    
    with torch.no_grad():
        for lr_img, hr_img in val_loader:
            if sample_idx >= num_samples:
                break
            
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate SR images
            sr_bicubic = bicubic(lr_img)
            sr_ours = generator(lr_img)
            
            # Calculate absolute error maps
            error_bicubic = torch.abs(sr_bicubic - hr_img).mean(dim=1, keepdim=True)  # Average across RGB
            error_ours = torch.abs(sr_ours - hr_img).mean(dim=1, keepdim=True)
            
            # Visualize
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            
            # Row 1: Ground Truth, Bicubic SR, Ours SR
            hr_np = hr_img[0].cpu().permute(1, 2, 0).numpy()
            hr_np = np.clip(hr_np, 0, 1)
            axes[0, 0].imshow(hr_np)
            axes[0, 0].set_title('Ground Truth (HR)', fontsize=11, fontweight='bold')
            axes[0, 0].axis('off')
            
            bicubic_np = sr_bicubic[0].cpu().permute(1, 2, 0).numpy()
            bicubic_np = np.clip(bicubic_np, 0, 1)
            axes[0, 1].imshow(bicubic_np)
            axes[0, 1].set_title('Bicubic SR', fontsize=11, fontweight='bold')
            axes[0, 1].axis('off')
            
            ours_np = sr_ours[0].cpu().permute(1, 2, 0).numpy()
            ours_np = np.clip(ours_np, 0, 1)
            axes[0, 2].imshow(ours_np)
            axes[0, 2].set_title('RFB-ESRGAN SR (Ours)', fontsize=11, fontweight='bold')
            axes[0, 2].axis('off')
            
            # Row 2: Error maps
            axes[1, 0].axis('off')  # Empty cell
            
            error_bicubic_np = error_bicubic[0, 0].cpu().numpy()
            mae_bicubic = error_bicubic_np.mean()
            im1 = axes[1, 1].imshow(error_bicubic_np, cmap='hot', vmin=0, vmax=0.3)
            axes[1, 1].set_title(f'Bicubic Error Map\nMAE: {mae_bicubic:.4f}', fontsize=11, fontweight='bold')
            axes[1, 1].axis('off')
            plt.colorbar(im1, ax=axes[1, 1], fraction=0.046, pad=0.04)
            
            error_ours_np = error_ours[0, 0].cpu().numpy()
            mae_ours = error_ours_np.mean()
            im2 = axes[1, 2].imshow(error_ours_np, cmap='hot', vmin=0, vmax=0.3)
            axes[1, 2].set_title(f'RFB-ESRGAN Error Map\nMAE: {mae_ours:.4f}', fontsize=11, fontweight='bold', color='green')
            axes[1, 2].axis('off')
            plt.colorbar(im2, ax=axes[1, 2], fraction=0.046, pad=0.04)
            
            plt.tight_layout()
            fig_path = os.path.join(save_dir, f'error_map_sample_{sample_idx+1}.png')
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            print(f"✓ Saved: error_map_sample_{sample_idx+1}.png")
            plt.close()
            
            sample_idx += 1
    
    print(f"\n✓ Difference maps saved to: {save_dir}")


print("✓ Difference maps visualization function defined!")

In [ ]:
# ========== 7. MODEL CONVERGENCE ANALYSIS ==========

def analyze_model_convergence(generator, val_loader, device, save_dir):
    """Analyze prediction quality distribution and stability"""
    os.makedirs(save_dir, exist_ok=True)
    
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    
    psnr_list = []
    ssim_list = []
    improvements_psnr = []
    improvements_ssim = []
    
    generator.eval()
    
    print("\n📊 Analyzing model convergence and stability...")
    
    with torch.no_grad():
        for lr_img, hr_img in tqdm(val_loader, desc="Analyzing", ncols=80):
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Our model
            sr_ours = generator(lr_img)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            
            # Bicubic baseline
            sr_bicubic = bicubic(lr_img)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            
            # Store metrics
            psnr_list.append(metrics_ours['psnr'])
            ssim_list.append(metrics_ours['ssim'])
            
            # Store improvements
            improvements_psnr.append(metrics_ours['psnr'] - metrics_bicubic['psnr'])
            improvements_ssim.append(metrics_ours['ssim'] - metrics_bicubic['ssim'])
    
    # ========== Figure 1: Metric Distributions ==========
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # PSNR histogram
    axes[0, 0].hist(psnr_list, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(np.mean(psnr_list), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(psnr_list):.2f} dB')
    axes[0, 0].set_xlabel('PSNR (dB)', fontsize=11)
    axes[0, 0].set_ylabel('Frequency', fontsize=11)
    axes[0, 0].set_title('PSNR Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # SSIM histogram
    axes[0, 1].hist(ssim_list, bins=50, color='seagreen', alpha=0.7, edgecolor='black')
    axes[0, 1].axvline(np.mean(ssim_list), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(ssim_list):.4f}')
    axes[0, 1].set_xlabel('SSIM', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('SSIM Distribution', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # PSNR improvement distribution
    axes[1, 0].hist(improvements_psnr, bins=50, color='purple', alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(np.mean(improvements_psnr), color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: +{np.mean(improvements_psnr):.2f} dB')
    axes[1, 0].set_xlabel('ΔPSNR vs. Bicubic (dB)', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title('PSNR Improvement Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    # SSIM improvement distribution
    axes[1, 1].hist(improvements_ssim, bins=50, color='coral', alpha=0.7, edgecolor='black')
    axes[1, 1].axvline(np.mean(improvements_ssim), color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: +{np.mean(improvements_ssim):.4f}')
    axes[1, 1].set_xlabel('ΔSSIM vs. Bicubic', fontsize=11)
    axes[1, 1].set_ylabel('Frequency', fontsize=11)
    axes[1, 1].set_title('SSIM Improvement Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'convergence_analysis.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: convergence_analysis.png")
    plt.close()
    
    # ========== Figure 2: Performance Trends ==========
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # PSNR trend
    axes[0].plot(psnr_list, alpha=0.6, linewidth=0.8, color='steelblue')
    axes[0].axhline(np.mean(psnr_list), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[0].fill_between(range(len(psnr_list)), 
                         np.mean(psnr_list) - np.std(psnr_list), 
                         np.mean(psnr_list) + np.std(psnr_list), 
                         alpha=0.2, color='red', label='±1 Std Dev')
    axes[0].set_xlabel('Sample Index', fontsize=11)
    axes[0].set_ylabel('PSNR (dB)', fontsize=11)
    axes[0].set_title('PSNR Across Validation Set', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # SSIM trend
    axes[1].plot(ssim_list, alpha=0.6, linewidth=0.8, color='seagreen')
    axes[1].axhline(np.mean(ssim_list), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[1].fill_between(range(len(ssim_list)), 
                         np.mean(ssim_list) - np.std(ssim_list), 
                         np.mean(ssim_list) + np.std(ssim_list), 
                         alpha=0.2, color='red', label='±1 Std Dev')
    axes[1].set_xlabel('Sample Index', fontsize=11)
    axes[1].set_ylabel('SSIM', fontsize=11)
    axes[1].set_title('SSIM Across Validation Set', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'performance_trends.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: performance_trends.png")
    plt.close()
    
    # ========== Statistics Summary ==========
    print("\n" + "="*70)
    print("CONVERGENCE ANALYSIS SUMMARY")
    print("="*70)
    print(f"\nPSNR Statistics:")
    print(f"  Mean: {np.mean(psnr_list):.2f} dB")
    print(f"  Std Dev: {np.std(psnr_list):.2f} dB")
    print(f"  Min: {np.min(psnr_list):.2f} dB")
    print(f"  Max: {np.max(psnr_list):.2f} dB")
    print(f"  Median: {np.median(psnr_list):.2f} dB")
    
    print(f"\nSSIM Statistics:")
    print(f"  Mean: {np.mean(ssim_list):.4f}")
    print(f"  Std Dev: {np.std(ssim_list):.4f}")
    print(f"  Min: {np.min(ssim_list):.4f}")
    print(f"  Max: {np.max(ssim_list):.4f}")
    print(f"  Median: {np.median(ssim_list):.4f}")
    
    print(f"\nImprovement Statistics (vs. Bicubic):")
    print(f"  ΔPSNR Mean: +{np.mean(improvements_psnr):.2f} dB")
    print(f"  ΔPSNR Std Dev: {np.std(improvements_psnr):.2f} dB")
    print(f"  ΔSSIM Mean: +{np.mean(improvements_ssim):.4f}")
    print(f"  ΔSSIM Std Dev: {np.std(improvements_ssim):.4f}")
    
    # Coefficient of Variation (measure of stability)
    cv_psnr = (np.std(psnr_list) / np.mean(psnr_list)) * 100
    cv_ssim = (np.std(ssim_list) / np.mean(ssim_list)) * 100
    
    print(f"\nStability Metrics (Coefficient of Variation):")
    print(f"  PSNR CV: {cv_psnr:.2f}% (lower is more stable)")
    print(f"  SSIM CV: {cv_ssim:.2f}% (lower is more stable)")
    
    print(f"\n✓ Convergence analysis completed and saved to: {save_dir}")


print("✓ Convergence analysis function defined!")

In [ ]:
# ========== 8. FAILURE CASE ANALYSIS ==========

def analyze_failure_cases(generator, val_loader, device, save_dir, num_worst=10):
    """Identify and visualize worst-performing samples"""
    os.makedirs(save_dir, exist_ok=True)
    
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    
    all_samples = []
    
    generator.eval()
    
    print("\n🔍 Identifying failure cases...")
    
    with torch.no_grad():
        for batch_idx, (lr_img, hr_img) in enumerate(tqdm(val_loader, desc="Scanning", ncols=80)):
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate SR
            sr_ours = generator(lr_img)
            sr_bicubic = bicubic(lr_img)
            
            # Calculate metrics
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            
            # Store sample info
            all_samples.append({
                'batch_idx': batch_idx,
                'lr': lr_img[0].cpu(),
                'hr': hr_img[0].cpu(),
                'sr_ours': sr_ours[0].cpu(),
                'sr_bicubic': sr_bicubic[0].cpu(),
                'psnr': metrics_ours['psnr'],
                'ssim': metrics_ours['ssim'],
                'improvement_psnr': metrics_ours['psnr'] - metrics_bicubic['psnr'],
                'improvement_ssim': metrics_ours['ssim'] - metrics_bicubic['ssim'],
            })
    
    # ========== Find Worst Cases ==========
    # Sort by PSNR (ascending)
    worst_psnr = sorted(all_samples, key=lambda x: x['psnr'])[:num_worst]
    
    # Sort by improvement over bicubic (ascending - least improvement or regression)
    worst_improvement = sorted(all_samples, key=lambda x: x['improvement_psnr'])[:num_worst]
    
    print(f"\n🔴 Found {num_worst} worst-performing samples")
    
    # ========== Visualize Worst PSNR Cases ==========
    fig, axes = plt.subplots(num_worst, 4, figsize=(16, 4*num_worst))
    
    if num_worst == 1:
        axes = axes.reshape(1, -1)
    
    for i, sample in enumerate(worst_psnr):
        # LR
        lr_np = F.interpolate(sample['lr'].unsqueeze(0), scale_factor=8, mode='nearest')[0].permute(1, 2, 0).numpy()
        lr_np = np.clip(lr_np, 0, 1)
        axes[i, 0].imshow(lr_np)
        axes[i, 0].set_title(f'LR Input (Sample #{sample["batch_idx"]})', fontsize=9)
        axes[i, 0].axis('off')
        
        # HR
        hr_np = sample['hr'].permute(1, 2, 0).numpy()
        hr_np = np.clip(hr_np, 0, 1)
        axes[i, 1].imshow(hr_np)
        axes[i, 1].set_title('HR Ground Truth', fontsize=9)
        axes[i, 1].axis('off')
        
        # Bicubic
        bicubic_np = sample['sr_bicubic'].permute(1, 2, 0).numpy()
        bicubic_np = np.clip(bicubic_np, 0, 1)
        axes[i, 2].imshow(bicubic_np)
        axes[i, 2].set_title('Bicubic SR', fontsize=9)
        axes[i, 2].axis('off')
        
        # Ours
        ours_np = sample['sr_ours'].permute(1, 2, 0).numpy()
        ours_np = np.clip(ours_np, 0, 1)
        axes[i, 3].imshow(ours_np)
        axes[i, 3].set_title(f'Ours (PSNR: {sample["psnr"]:.2f} dB)', fontsize=9, color='red')
        axes[i, 3].axis('off')
    
    plt.suptitle(f'Top {num_worst} Worst PSNR Cases', fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'worst_psnr_cases.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: worst_psnr_cases.png")
    plt.close()
    
    # ========== Visualize Cases with Least Improvement ==========
    fig, axes = plt.subplots(num_worst, 4, figsize=(16, 4*num_worst))
    
    if num_worst == 1:
        axes = axes.reshape(1, -1)
    
    for i, sample in enumerate(worst_improvement):
        # LR
        lr_np = F.interpolate(sample['lr'].unsqueeze(0), scale_factor=8, mode='nearest')[0].permute(1, 2, 0).numpy()
        lr_np = np.clip(lr_np, 0, 1)
        axes[i, 0].imshow(lr_np)
        axes[i, 0].set_title(f'LR Input (Sample #{sample["batch_idx"]})', fontsize=9)
        axes[i, 0].axis('off')
        
        # HR
        hr_np = sample['hr'].permute(1, 2, 0).numpy()
        hr_np = np.clip(hr_np, 0, 1)
        axes[i, 1].imshow(hr_np)
        axes[i, 1].set_title('HR Ground Truth', fontsize=9)
        axes[i, 1].axis('off')
        
        # Bicubic
        bicubic_np = sample['sr_bicubic'].permute(1, 2, 0).numpy()
        bicubic_np = np.clip(bicubic_np, 0, 1)
        axes[i, 2].imshow(bicubic_np)
        axes[i, 2].set_title('Bicubic SR', fontsize=9)
        axes[i, 2].axis('off')
        
        # Ours
        ours_np = sample['sr_ours'].permute(1, 2, 0).numpy()
        ours_np = np.clip(ours_np, 0, 1)
        axes[i, 3].imshow(ours_np)
        axes[i, 3].set_title(f'Ours (Δ: {sample["improvement_psnr"]:+.2f} dB)', fontsize=9, color='red')
        axes[i, 3].axis('off')
    
    plt.suptitle(f'Top {num_worst} Cases with Least Improvement vs. Bicubic', fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    fig_path = os.path.join(save_dir, 'worst_improvement_cases.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: worst_improvement_cases.png")
    plt.close()
    
    # ========== Statistics for Failure Cases ==========
    print("\n" + "="*70)
    print("FAILURE CASE ANALYSIS")
    print("="*70)
    
    print(f"\nWorst PSNR Cases:")
    for i, sample in enumerate(worst_psnr[:5]):
        print(f"  #{i+1}: Batch {sample['batch_idx']}, PSNR={sample['psnr']:.2f} dB, SSIM={sample['ssim']:.4f}")
    
    print(f"\nCases with Least Improvement over Bicubic:")
    for i, sample in enumerate(worst_improvement[:5]):
        print(f"  #{i+1}: Batch {sample['batch_idx']}, Δ={sample['improvement_psnr']:+.2f} dB (PSNR={sample['psnr']:.2f} dB)")
    
    # Check for regressions
    regressions = [s for s in all_samples if s['improvement_psnr'] < 0]
    print(f"\n⚠️  Regression Cases (worse than Bicubic): {len(regressions)} samples")
    
    if len(regressions) > 0:
        print(f"  Average regression: {np.mean([s['improvement_psnr'] for s in regressions]):.2f} dB")
    
    print(f"\n✓ Failure case analysis completed and saved to: {save_dir}")


print("✓ Failure case analysis function defined!")

In [ ]:
# ========== 9. EXECUTE COMPREHENSIVE EVALUATION ==========

print("\n" + "="*80)
print(" " * 20 + "🚀 COMPREHENSIVE MODEL EVALUATION")
print("="*80)

# Create evaluation output directory
eval_output_dir = '/kaggle/working/RFB-ESRGAN-Output/evaluation'
os.makedirs(eval_output_dir, exist_ok=True)

print(f"\n📁 Evaluation results will be saved to: {eval_output_dir}")

# ========== 1. COMPARATIVE EVALUATION ==========
print("\n" + "="*80)
print("PHASE 1: COMPARATIVE EVALUATION")
print("="*80)

comparison_table, results, inference_times = comparative_evaluation(
    generator=generator,
    val_loader=val_loader,
    device=device,
    num_samples=100  # Evaluate on 100 samples
)

# ========== 2. VISUALIZATION FUNCTIONS ==========
print("\n" + "="*80)
print("PHASE 2: COMPARATIVE VISUALIZATIONS")
print("="*80)

viz_dir = os.path.join(eval_output_dir, 'comparisons')
create_comparison_visualizations(results, inference_times, save_dir=viz_dir)

# Log visualizations to WandB
print("\n📤 Uploading visualizations to WandB...")
for img_name in ['metrics_comparison.png', 'metrics_distribution.png', 'radar_chart.png', 'quality_speed_tradeoff.png']:
    img_path = os.path.join(viz_dir, img_name)
    if os.path.exists(img_path):
        wandb.log({f"eval/{img_name.replace('.png', '')}": wandb.Image(img_path)})
        print(f"  ✓ Uploaded: {img_name}")

# ========== 3. VISUAL QUALITY COMPARISON ==========
print("\n" + "="*80)
print("PHASE 3: VISUAL QUALITY COMPARISON")
print("="*80)

quality_dir = os.path.join(eval_output_dir, 'quality_samples')
visualize_quality_comparison(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=quality_dir,
    num_samples=5  # Generate 5 comparison samples
)

# Log quality samples to WandB
print("\n📤 Uploading quality samples to WandB...")
for i in range(1, 6):
    img_path = os.path.join(quality_dir, f'comparison_sample_{i}.png')
    if os.path.exists(img_path):
        wandb.log({f"eval/quality_sample_{i}": wandb.Image(img_path)})

# ========== 4. DIFFERENCE MAPS ==========
print("\n" + "="*80)
print("PHASE 4: DIFFERENCE MAPS")
print("="*80)

diff_dir = os.path.join(eval_output_dir, 'difference_maps')
visualize_difference_maps(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=diff_dir,
    num_samples=5
)

# Log difference maps to WandB
print("\n📤 Uploading difference maps to WandB...")
for i in range(1, 6):
    img_path = os.path.join(diff_dir, f'error_map_sample_{i}.png')
    if os.path.exists(img_path):
        wandb.log({f"eval/error_map_{i}": wandb.Image(img_path)})

# ========== 5. CONVERGENCE ANALYSIS ==========
print("\n" + "="*80)
print("PHASE 5: CONVERGENCE ANALYSIS")
print("="*80)

convergence_dir = os.path.join(eval_output_dir, 'convergence')
analyze_model_convergence(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=convergence_dir
)

# Log convergence plots to WandB
print("\n📤 Uploading convergence analysis to WandB...")
for img_name in ['convergence_analysis.png', 'performance_trends.png']:
    img_path = os.path.join(convergence_dir, img_name)
    if os.path.exists(img_path):
        wandb.log({f"eval/{img_name.replace('.png', '')}": wandb.Image(img_path)})

# ========== 6. FAILURE CASE ANALYSIS ==========
print("\n" + "="*80)
print("PHASE 6: FAILURE CASE ANALYSIS")
print("="*80)

failure_dir = os.path.join(eval_output_dir, 'failure_cases')
analyze_failure_cases(
    generator=generator,
    val_loader=val_loader,
    device=device,
    save_dir=failure_dir,
    num_worst=10
)

# Log failure cases to WandB
print("\n📤 Uploading failure case analysis to WandB...")
for img_name in ['worst_psnr_cases.png', 'worst_improvement_cases.png']:
    img_path = os.path.join(failure_dir, img_name)
    if os.path.exists(img_path):
        wandb.log({f"eval/{img_name.replace('.png', '')}": wandb.Image(img_path)})

# ========== FINAL SUMMARY ==========
print("\n" + "="*80)
print(" " * 20 + "✅ EVALUATION COMPLETE")
print("="*80)

print(f"\n📊 Summary of Evaluation:")
print(f"  • Comparative evaluation: 5 models compared on 100 samples")
print(f"  • Visualizations: 12+ plots generated")
print(f"  • Quality samples: 5 side-by-side comparisons")
print(f"  • Difference maps: 5 error heatmaps")
print(f"  • Convergence analysis: Distribution and trend plots")
print(f"  • Failure analysis: Top 10 worst-performing cases identified")

print(f"\n📁 All results saved to: {eval_output_dir}")
print(f"📤 All metrics and visualizations uploaded to WandB")

print("\n🎉 Comprehensive evaluation completed successfully!")
print("="*80)